# 01 · LangChain básico — pregunta → SQL → respuesta

El flujo más simple con LangChain. Este cuaderno hace **una sola pasada**: convierte una pregunta en lenguaje natural a SQL, la ejecuta y la convierte de vuelta a texto.

**Objetivos:**

1. Entender las tres piezas mínimas de LangChain: `ChatModel`, `PromptTemplate`, `OutputParser`.
2. Componerlas con el operador `|` (LCEL, LangChain Expression Language).
3. Ejecutar un LLM local vs un LLM falso (para que el cuaderno corra sin Ollama).

> **Antes de correr:** asegúrate de haber cargado la BD del proyecto 06 (`cd ../../06-ingesta-eventos-ecommerce && make pipeline`). Si no la tienes, igual puedes correr este cuaderno con una BD de prueba en memoria.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("proyecto:", PROJECT_ROOT)

proyecto: /Users/david.palacio/Documents/data-projects-lab/projects/07-chat-con-tu-bd


## 1 · Preparamos una BD de prueba

Para que el cuaderno sea autosuficiente, montamos una BD SQLite pequeña en memoria. Si tienes la del 06, puedes saltarte esta celda y apuntar `DATABASE_URL` a ella.

In [2]:
import os
import tempfile
from sqlalchemy import create_engine, text

db_path = Path(tempfile.gettempdir()) / "demo_chat_bd.db"
os.environ["DATABASE_URL"] = f"sqlite:///{db_path}"

engine = create_engine(os.environ["DATABASE_URL"], future=True)
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS productos"))
    conn.execute(text("DROP TABLE IF EXISTS ventas"))
    conn.execute(text("CREATE TABLE productos (id INTEGER PRIMARY KEY, nombre TEXT, categoria TEXT, precio REAL)"))
    conn.execute(text("CREATE TABLE ventas (id INTEGER PRIMARY KEY, producto_id INTEGER, cantidad INTEGER, fecha DATE)"))
    for i, (n, cat, p) in enumerate(
        [("Auriculares", "Electronica", 120), ("Teclado", "Electronica", 80),
         ("Mochila", "Accesorios", 60), ("Taza", "Hogar", 15), ("Libro", "Libros", 25)],
        start=1,
    ):
        conn.execute(text("INSERT INTO productos VALUES (:i, :n, :c, :p)"), {"i": i, "n": n, "c": cat, "p": p})
    for i in range(1, 11):
        conn.execute(text("INSERT INTO ventas VALUES (:i, :pid, :c, DATE('now', '-' || :d || ' days'))"),
                     {"i": i, "pid": (i % 5) + 1, "c": i, "d": i})

# Hay que limpiar el cache del engine en chat_bd para que recoja la nueva URL
from chat_bd import config, sql_tools
config.DATABASE_URL = os.environ["DATABASE_URL"]
sql_tools.DATABASE_URL = os.environ["DATABASE_URL"]
sql_tools.crear_engine.cache_clear()

print("BD lista en:", db_path)

BD lista en: /var/folders/l4/1t58ff412ss3dr4zyn2_6wwhch78n1/T/demo_chat_bd.db


## 2 · El LLM: real (Ollama) o falso (para que el cuaderno siempre corra)

La factory `obtener_llm()` decide según `LLM_PROVIDER`. En este cuaderno forzamos `fake` para que el notebook se ejecute sin depender de Ollama. En producción cambiarías a `ollama` (o `hf`).

In [3]:
from chat_bd.llm import obtener_llm

# LLM falso: le damos las respuestas que esperamos (en orden de invocacion).
llm = obtener_llm(
    proveedor="fake",
    respuestas_fake=[
        "SELECT nombre, precio FROM productos ORDER BY precio DESC LIMIT 3",
        "Los 3 productos mas caros son Auriculares (120), Teclado (80) y Mochila (60).",
    ],
)

# Para probar con Ollama de verdad (requiere `ollama serve` corriendo):
# llm = obtener_llm(proveedor="ollama")

print(type(llm).__name__)

FakeListChatModel


## 3 · El schema: lo que el LLM necesita saber

Un LLM que no conoce el schema **inventa** nombres de tabla y columna. La pieza clave es inyectar un resumen del schema en el prompt.

In [4]:
from chat_bd.sql_tools import schema_como_texto

print(schema_como_texto())

- productos(id INTEGER, nombre TEXT, categoria TEXT, precio REAL)  -- 5 filas
- ventas(id INTEGER, producto_id INTEGER, cantidad INTEGER, fecha DATE)  -- 10 filas


## 4 · La cadena con LCEL: `prompt | llm | parser`

LCEL (LangChain Expression Language) encadena componentes con `|`. La salida de uno es la entrada del siguiente.

In [5]:
from langchain_core.output_parsers import StrOutputParser
from chat_bd.prompts import prompt_sql

cadena_sql = prompt_sql | llm | StrOutputParser()

pregunta = "cuales son los 3 productos mas caros?"
sql = cadena_sql.invoke({"pregunta": pregunta, "schema": schema_como_texto()})
print("SQL generado:")
print(sql)

SQL generado:
SELECT nombre, precio FROM productos ORDER BY precio DESC LIMIT 3


In [6]:
# Ejecutamos el SQL contra la BD, con la validacion de seguridad.
from chat_bd.sql_tools import ejecutar_sql_lectura

df = ejecutar_sql_lectura(sql)
df

,nombre,precio
0,Auriculares,120.0
1,Teclado,80.0
2,Mochila,60.0


In [7]:
# Segunda cadena: con el resultado, el LLM lo convierte en respuesta natural.
from chat_bd.prompts import prompt_respuesta

cadena_resp = prompt_respuesta | llm | StrOutputParser()
respuesta = cadena_resp.invoke({"pregunta": pregunta, "sql": sql, "resultado": df.to_string(index=False)})
print(respuesta)

Los 3 productos mas caros son Auriculares (120), Teclado (80) y Mochila (60).


## 5 · Todo junto: `chain_simple.ejecutar`

El modulo `chain_simple` tiene lo mismo empaquetado en una funcion. Lo que escribiste arriba es lo que hay dentro.

In [8]:
from chat_bd.chain_simple import ejecutar

# Recreamos el LLM porque el anterior ya consumio sus respuestas.
llm2 = obtener_llm(
    proveedor="fake",
    respuestas_fake=[
        "SELECT COUNT(*) AS total FROM productos",
        "Hay 5 productos en total.",
    ],
)

r = ejecutar("cuantos productos hay?", llm=llm2)
print("SQL:     ", r.sql)
print("Resultado:", r.resultado.to_dict(orient="records"))
print("Respuesta:", r.respuesta)

SQL:      SELECT COUNT(*) AS total FROM productos
Resultado: [{'total': 5}]
Respuesta: Hay 5 productos en total.


## Cierre

- **LangChain** te da los ladrillos: `ChatModel`, `PromptTemplate`, `OutputParser`.
- **LCEL** (`|`) los compone en una tubería.
- **El schema es parte del prompt**: sin él, el LLM alucina nombres de tabla.
- **Un LLM falso** te deja testear sin Ollama ni red.

**Limitación:** si el LLM genera SQL mal formado (p. ej. refiere una tabla inexistente), esta cadena **falla y punto**. Hacer que se corrija es el caso de uso del siguiente notebook: **LangGraph**.